# EDA


## 0) Wstęp

Wczytanie odpoweidnich bibliotek

In [62]:
import sqlite3                  # Do pracy z bazą danych
import pandas as pd             # DO pracy z danymi
import plotly.express as px     # Ostatnie dwa pliki służą do interaktywnych wykresów
import plotly.io as pio
import sys                      # Za cwhile wczytamy plik config z rzeczami stałymi
import os
from scipy.stats import chi2_contingency # Do testów statyyzcnych

In [49]:
# prawdzamy gdzie opeobenie działamy
current_dir = os.getcwd()

if current_dir.endswith('notebooks'):
    # Jeśli jesteśmy w 'notebooks', wychodzimy poziom wyżej
    sys.path.append(os.path.abspath('../src'))
    db_path = '../hospital_data.db'
else:
    # Jeśli VS Code odpalił nas z głównego folderu 'health_model'
    sys.path.append(os.path.abspath('./src'))
    db_path = './hospital_data.db'

# Na koniec wczytujemy plik ze stałymi
import config

## 1) Wczytanie danych

Na start lecimy z bazą

In [50]:
pio.templates.default = config.PLOT_CONFIG['template']
conn = sqlite3.connect(db_path)

Wczytujemy wszytkie dane

In [51]:
query = """
    SELECT 
        p.*,
        a.description AS admission_type_desc,
        d.description AS discharge_desc,
        s.description AS admission_source_desc
    FROM patients p
    LEFT JOIN admission_type a ON p.admission_type_id = a.admission_type_id
    LEFT JOIN discharge_disposition d ON p.discharge_disposition_id = d.discharge_disposition_id
    LEFT JOIN admission_source s ON p.admission_source_id = s.admission_source_id;
"""

df = pd.read_sql_query(query, conn)
conn.close()

In [52]:
print(f"Liczba wierszy: {df.shape[0]}\nLiczba kolumn: {df.shape[1]}")

Liczba wierszy: 101766
Liczba kolumn: 53


Zróbmy szybkie podejrzenie danych

In [53]:
df

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,admission_type_desc,discharge_desc,admission_source_desc
0,2278392,8222157,Caucasian,Female,[0-10),None,6,25,1,1,...,No,No,No,No,No,No,NO,None,Not Mapped,Physician Referral
1,149190,55629189,Caucasian,Female,[10-20),None,1,1,7,3,...,No,No,No,No,Ch,Yes,>30,Emergency,Discharged to home,Emergency Room
2,64410,86047875,AfricanAmerican,Female,[20-30),None,1,1,7,2,...,No,No,No,No,No,Yes,NO,Emergency,Discharged to home,Emergency Room
3,500364,82442376,Caucasian,Male,[30-40),None,1,1,7,2,...,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room
4,16680,42519267,Caucasian,Male,[40-50),None,1,1,7,1,...,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),None,1,3,7,3,...,No,No,No,No,Ch,Yes,>30,Emergency,Discharged/transferred to SNF,Emergency Room
101762,443847782,74694222,AfricanAmerican,Female,[80-90),None,1,4,5,5,...,No,No,No,No,No,Yes,NO,Emergency,Discharged/transferred to ICF,Transfer from a Skilled Nursing Facility (SNF)
101763,443854148,41088789,Caucasian,Male,[70-80),None,1,1,7,1,...,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room
101764,443857166,31693671,Caucasian,Female,[80-90),None,2,3,7,10,...,No,No,No,No,Ch,Yes,NO,Urgent,Discharged/transferred to SNF,Emergency Room


## 2) Braki Danych

#### Cechy z największsą iloscią braków

Zaczniemy od braków dnaych

In [54]:
# Obliczmay jaki jest procent brakujących danych dla każdej kolumny
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False).reset_index()
missing_data.columns = ['Kolumna', 'Liczba braków']
missing_data['Procent'] = (missing_data['Liczba braków'] / len(df)) * 100

Teraz zrobimy interaktywny wykres pokazujący ile jest tych braków

In [55]:
# Rysowanie interaktywnego wykresu słupkowego za pomocą Plotly Express
fig = px.bar(
    missing_data, 
    x='Kolumna', 
    y='Procent',
    title='Procent brakujących danych w poszczególnych kolumnach',
    # Pokazuje wartości na czubku słupków z jednym miejscem po przecinku
    text_auto='.1f', 
    labels={'Procent': 'Braki (%)', 'Kolumna': 'Nazwa cechy'},
    # Koloruje słupki na podstawie wielkości braku
    color='Procent',
     # Czerwony kolor ostrzega nas o problemie
    color_continuous_scale='Reds')

# Poprawienie czytelności osi X - obrót lierek o kąt 45 stopni zamiast domyślnego
fig.update_layout(xaxis_tickangle=-45)

fig.show()

#### Współwystępownie braków dnaych

In [ ]:
# Wyciągamy nazwy 5 kolumn z największą liczbą braków
top_5_missing = missing_data.head(5)['Kolumna'].tolist()

# Tworzymy tabelę pomocniczą, gdzie 1 oznacza BRAK danych, a 0 oznacza coś innego
df_missing_flags = df[top_5_missing].isnull().astype(int)

# Obliczamy macierz korelacji Pearsona 
missing_corr = df_missing_flags.corr()

# Rysujemy interaktywną Heatmapę w Plotly
fig_missing_corr = px.imshow(
    missing_corr,
    text_auto=".2f",                                            # Pokaż wartości z dwoma miejscami po przecinku
    #'''
    #zmien to pozniej
    #'''
    color_continuous_scale='RdBu_r',                            # niebieski = dodatnia korelacja, czerwony = ujemna)
    zmin=-1, zmax=1,                                            # Skala korelacji zawsze od -1 do 1
    title="Współwystępowanie braków danych",
    labels=dict(color="Korelacja")
)

# Upiększanie wyglądu za pomocą naszego pliku stałego
fig_missing_corr.update_layout(
    font=dict(family=config.PLOT_CONFIG['font_family'], size=config.PLOT_CONFIG['label_font_size']),
    title_font_size=config.PLOT_CONFIG['title_font_size'],
    width=config.PLOT_CONFIG['width'],
    height=config.PLOT_CONFIG['height']
)

fig_missing_corr.show()

Teraz jeszcze spojrzymy na nakładanie się braków danych

In [58]:
print(pd.crosstab(
    df['weight'].isnull(), 
    df['max_glu_serum'].isnull(), 
    rownames=['Brak Wagi'], 
    colnames=['Brak Badania Krwi'],
    normalize='all'
) * 100)

Brak Badania Krwi     False      True 
Brak Wagi                             
False              0.000000   3.141521
True               5.253228  91.605251


Z tej tabeli idą nsatępujące wnioski:
 - **92%** pacjentów ma brak zarówno i w jednej i w drugiej kolumnie $\rightarrow$ bardzo dużo
 - **około 0%** pacjentów ma obie kolumny uzupełnione
 
 Czyli wniosek jest taki obie te kolumny są śmieciowe i do wywalenia

#### Dalej z brakami


Rozważymy narazie jeden przypadek dokładniej, a później sprawdzimy reszte

In [61]:
# Tworzymy tymczasową kolumnę, która mówi tylko: Czy brakuje specjalizacji?
df_missing_analysis = df.copy()
df_missing_analysis['Brak_Specjalizacji'] = df_missing_analysis['medical_specialty'].isnull()

# Grupujemy dane, żeby sprawdzić jak rozkłada się powrót do szpitala
# w zależności od tego, czy brakuje specjalizacji
missing_target_crosstab = pd.crosstab(
    df_missing_analysis['Brak_Specjalizacji'], 
    df_missing_analysis['readmitted'], 
    normalize='index'                   # Normalizujemy, żeby uzyskać procenty w każdym wierszu
) * 100

missing_target_crosstab = missing_target_crosstab.reset_index()
missing_target_crosstab['Brak_Specjalizacji'] = missing_target_crosstab['Brak_Specjalizacji'].map({True: 'Tak (Brak danych)', False: 'Nie (Dane obecne)'})

# Przekształcenie tabeli do formatu długiego (melt), idealnego dla Plotly
missing_target_melted = missing_target_crosstab.melt(
    id_vars='Brak_Specjalizacji', 
    var_name='Status Powrotu', 
    value_name='Procent'
)

# Rysujemy piękny wykres skumulowany
fig_missing = px.bar(
    missing_target_melted,
    x='Brak_Specjalizacji',
    y='Procent',
    color='Status Powrotu',
    title='Czy brak danych o specjalizacji lekarza wpływa na powrót do szpitala?',
    text_auto='.1f',
    color_discrete_sequence=[config.COLORS['primary'], config.COLORS['secondary'], config.COLORS['danger']],
    barmode='stack'
)

fig_missing.update_layout(
    font=dict(family=config.PLOT_CONFIG['font_family'], size=config.PLOT_CONFIG['label_font_size']),
    title_font_size=config.PLOT_CONFIG['title_font_size'],
    width=config.PLOT_CONFIG['width'],
    height=config.PLOT_CONFIG['height'],
    xaxis_title='Czy brakuje danych o specjalizacji lekarza?',
    yaxis_title='Procent pacjentów (%)'
)

fig_missing.show()

Tutaj mogłoby się wydawawać, że różnice są pomijalne ale przy tak dużej ilosci danych to może nie być ten case.

 Aby sie upewnic należy wykonać test statystyczny - test zgodności $\chi^2$

In [ ]:
# Do testu statystycznego musimy podać liczby a nie procenty
contingency_table = pd.crosstab(
    df['medical_specialty'].isnull(), 
    df['readmitted']
)

print("Tabela kontyngencji z liczbami:")
print(contingency_table)
print("-" * 40)

# Wykonujemy test Chi-Kwadrat
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

# Wyświetlamy wynik z interpretacją
print(f"Wartość p-value: {p_value}")

if p_value < 0.05:
    print("Różnica jest wysoce istotna statystycznie (Brak przypadku)")
else:
    print("Różnica nie jest istotna statystycznie (Przypadek)")

Tabela kontyngencji z liczbami:
readmitted          <30    >30     NO
medical_specialty                    
False              5576  17329  28912
True               5781  18216  25952
----------------------------------------
Wartość p-value: 1.4027199576215145e-33
WERDYKT: Różnica JEST wysoce istotna statystycznie! (Brak przypadku)


Wyszło nam prawie zerowe p-value więc odrzucamy $H_0$ ,czyli przyjmujemy zatem, że te dane to nie przypadek i relanie jest jakaś zależność między nimi

Zatem już wiemy z testu, że jest jakaś zależność między tymi dwoma rzeczami, a z wykresu wyżej już możemy powidzieć, że brak danych w specjalizacji lekarza wpływa na większą szanse powrotu do lekarza. 

Zatem **powiniśmy dodac kolumne** mówiącą o ty m czy w tym miesjcu jest brak czy nie ma braku

Sprawdźmy teraz jeszcze czy dla drugiej połowy gdy mamy specjalizacje, to czy ta specjalziacja ma znaczenioe (istotność zmiennej)

In [ ]:
# Bierzemy tylko dane ze specjalizacją
df_present = df[df['medical_specialty'].notnull()].copy()

# Wybieramy 10 najczęstszych specjalizacji
top_specialties = df_present['medical_specialty'].value_counts().nlargest(10).index
df_top_spec = df_present[df_present['medical_specialty'].isin(top_specialties)]

# Tworzymy zestawienie specjalizacja vs powrót do szpitala
spec_target = pd.crosstab(
    df_top_spec['medical_specialty'], 
    df_top_spec['readmitted'], 
    normalize='index'
).reset_index()

# Wykres skumulowany
fig_spec = px.bar(
    spec_target.melt(id_vars='medical_specialty'),
    x='medical_specialty',
    y='value',
    color='readmitted',
    title='Wpływ konkretnej specjalizacji lekarza na powrót pacjenta',
    labels={'value': 'Procent (%)', 'medical_specialty': 'Specjalizacja'},
    color_discrete_map={'NO': config.COLORS['primary'], '>30': config.COLORS['secondary'], '<30': config.COLORS['danger']},
    barmode='stack'
)

fig_spec.update_layout(xaxis_tickangle=-45, width=config.PLOT_CONFIG['width'])
fig_spec.show()

Tutaj już widzimy, ze specjalizacja ma znacznie, bo dla różnych specjalizacji mamy różne wyniki, więc można już ostetcznie stwierdzić, że tę kolumnę zostawiamy ale dodajemy obok niejdrugą kolumnę mówiącą czy jest brak czy nie

#### Reszta kolumn

In [ ]:
# Lista kolumn z brakami (wybieramy te, które mają ich więcej niż 1% i mniej niż 90%)
cols_with_missing = missing_data[(missing_data['Procent'] > 1.0) & (90.0 > missing_data['Procent'])]['Kolumna'].tolist()

print(f"{'Kolumna':<25} | {'p-value':<12} | {'Czy istotna?'}")
print("-" * 55)

results = []

for col in cols_with_missing:
    # Tworzymy tabelę kontyngencji dla danej kolumny
    contingency = pd.crosstab(df[col].isnull(), df['readmitted'])
    
    # Test Chi-Kwadrat
    chi2, p, dof, ex = chi2_contingency(contingency)
    
    is_significant = "TAK" if p < 0.02 else "NIE"
    print(f"{col:<25} | {p:.2e} | {is_significant}")
    
    results.append({'Kolumna': col, 'p_value': p, 'Istotna': is_significant})

Kolumna                   | p-value      | Czy istotna?
-------------------------------------------------------
A1Cresult                 | 2.61e-12 | TAK
medical_specialty         | 1.40e-33 | TAK
payer_code                | 1.03e-03 | TAK
admission_source_desc     | 2.11e-02 | NIE
admission_type_desc       | 2.05e-30 | TAK
discharge_desc            | 3.41e-19 | TAK
race                      | 1.44e-41 | TAK
diag_3                    | 4.45e-33 | TAK


Z tego testu można określić, że możemy wywalić zmienna **admission_source_desc**

## 3) Analiza zmiennych numerycznych